In [ ]:
# This is not yet perfectly fitting to the figures in Zhu 2009 maybe you can find a mistake of mine :)
### Model
import mxlpy
from mxlpy import Model, Simulator,make_protocol, plot, scan
from mxlpy.fns import michaelis_menten_1s
import matplotlib.pyplot as plt

__all__ = ["get_model"]

def enzyme_atp_dependend(s1, ATP, V_max, K_ms1, K_mATP):
    return V_max*s1*ATP/((s1+K_ms1)*(ATP+K_mATP))

parameters= {
            # Vmax:
            "V1_max":3.78, #[mM/S] Zhu et al 2009
            "V2_max":11.75, #[mM/S] Zhu et al 2009
            "V3_max":5.04, #[mM/S] Zhu et al 2009
            "V4_max":3.05, #[mM/S] estimate by Zhu et al.
            "V5_max":3, #[mM/S] Zhu et al 2009
            "V6_max":0.1, #[mM/S] estimate by Zhu et al.
            "V13_max":8,
            # Km:
            "K_m1":1, #[mM] Zhu et al 2009
            "K_m21": 0.24, #[mM] Zhu et al 2009
            "K_m22":0.39, #[mM] Zhu et al 2009
            "K_m3":0.5, #[mM] Zhu et al 2009
            "K_m4":0.84, #[mM] Zhu et al 2009
            "K_m51":0.75, #[mM] Zhu et al 2009
            "K_m52":0.275, # [mM] not specified in the paper but fitted by us to match the figure
            "K_m6":5, #[mM] Zhu et al 2009
            "K_m131": 0.15, #[mM] Zhu et al 2009
            "K_m132":0.059,  #[mM] Zhu et al 2009
            "ATP":  0.2, # [mM]Not in Zhu et al 2009 but in the  reference [9] in the paper
         }

variables= {
            "RuBP" :2, # Zhu et al 2009
            "PGA":2.4, # Zhu et al 2009
            "DPGA":1, # Zhu et al 2009
            "Ru5P":1, # Zhu et al 2009
            "GAP":1,  # Zhu et al 2009
        }

def get_model() -> Model:
    """Simple Calvin Cycle Model developed by Zhu et al. (2009)."""
    m = Model()
    m.add_parameters(parameters)
    m.add_variables(variables)
    m.add_reaction(
        "v1",
        fn=michaelis_menten_1s,
        stoichiometry={"RuBP": -1, "PGA":2},
        args=["RuBP","V1_max", "K_m1",],
    )
    m.add_reaction(
        "v2",
        fn=enzyme_atp_dependend,
        stoichiometry={"PGA": -1, "DPGA":1 },
        args=["PGA","ATP","V2_max","K_m21", "K_m22"],
    )
    m.add_reaction(
        "v3",
        fn=michaelis_menten_1s,
        stoichiometry={"DPGA": -1, "GAP": 1},
        args=["DPGA", "V3_max", "K_m3"],
    )
    m.add_reaction(
        "v4",
        fn=michaelis_menten_1s,
        stoichiometry={"GAP": -1, "Ru5P": 0.6},
        args=["GAP", "V4_max", "K_m4"],
    )
    m.add_reaction(
        "v5",
        fn=enzyme_atp_dependend,
        stoichiometry={"PGA": -1,},
        args=["PGA", "ATP", "V5_max", "K_m51", "K_m52"],
    )
    m.add_reaction(
        "v6",
        fn=michaelis_menten_1s,
        stoichiometry={"GAP": -1, },
        args=["GAP", "V6_max", "K_m6"],
    )
    m.add_reaction(
        "v13",
        fn=enzyme_atp_dependend,
        stoichiometry={"Ru5P": -1,"RuBP": 1 },
        args=["Ru5P", "ATP", "V13_max", "K_m131", "K_m132"],
    )
    return m
    
    

In [ ]:
res = Simulator(get_model()).simulate(650).get_result().unwrap_or_err()

y_lim_max=[4,3,3,8,0.1]

fig, axes = plt.subplots(3,2, figsize=(9,10), )
axes = axes.flatten()
col = ["RuBP", "PGA","DPGA","GAP", "Ru5P"]
for i, col in enumerate(col):
    ax= axes[i]
    ax.plot(res.variables.index, res.variables[col], label=col)
    ax.set_xlim(left=0)
    ax.set_ylim(0, y_lim_max[i])
    ax.set_xlabel("time (s)")
    ax.set_ylabel("Concentration (mM)")
    ax.legend()

axes[5].plot(res.fluxes.index, res.fluxes["v1"]*33.33 # Here the carbon fixation is calculated by multiplying the flux with 33.33 (see Zhu et al. 2009) 
             , label="A")
axes[5].set_xlim(left=0)
axes[5].set_ylim(35,60)
axes[5].set_xlabel("time in s")
axes[5].set_ylabel("A (umol m-2 s-1)")
axes[5].legend()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

V1_max= 3.78
V2_max= 11.75
V3_max= 5.04
V4_max= 3.05 # estimate
V5_max= 3
V6_max= 0.1 #estimate
V13_max= 8 

V_max=["V1_max", "V3_max", "V4_max",  "V13_max", "V2_max","V5_max"]
V_max_variable =[V1_max, V3_max, V4_max, V13_max, V2_max, V5_max]

np_percent=np.array( [50, 100, 150, 200, 250, 300])
fig, axes=plt.subplots(3,2, figsize=(10,10))
axes= axes.flatten()

for i, V_max in enumerate(V_max):
    res = scan.steady_state(
        get_model(),
        to_scan=pd.DataFrame({V_max: V_max_variable[i]*np_percent/100}),
    )
    ax= axes[i]
    ax.scatter(np_percent, res.variables.loc[:, ["RuBP"]], label= ["RuBP"], marker="o")
    ax.scatter(np_percent, res.variables.loc[:, ["PGA"]], label= ["PGA",], marker="s")
    ax.scatter(np_percent, res.variables.loc[:, ["DPGA"]], label= ["DPGA",], marker="^")
    ax.scatter(np_percent, res.variables.loc[:, ["Ru5P"]], label= ["Ru5P",], marker="d")
    ax.set_xlabel(f"{V_max} (% of original value)")
    ax.set_ylabel("Concentration (mM)")
    ax.set_xlim(50, 300)

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5)
)
plt.show()